# The SDSS database: CasJobs and its functions

The Sloan Digital Sky Survey (SDSS) has imaged over a third of the sky and taken spectra of millions of objects. Everything it measured sits in a relational database of the kind you know from your SQL classes: tables of rows and columns, keys, joins and views. What is new is the scale and the purpose. The largest tables hold billions of rows, the schema was designed by astronomers for their own questions, and the database carries hundreds of functions of its own for positions on the sky, flags and cosmological distances.

This notebook introduces the database through the questions you meet when you use it such as how to reach it, how large it is, how to find the column you need, how its tables fit together, how to read its flags, which functions it offers, why some queries are fast and others never finish, and how to run a query that is too long to wait for? In this notebook we focus on one very specific patch of the sky in every section: the Coma cluster; a rich cluster of galaxies about $100$ megaparsecs away.

The three demo notebooks build on everything here; see the list at the end of this notebook.

Database ideas in this notebook:
- a database behind a web service, and the context a query runs in;
- the scale of the tables, and why queries travel to the data;
- the documentation of the schema, stored in the database itself;
- tables, views, keys and indexes;
- bit flags, coded values and markers of missing values;
- scalar and table-valued functions;
- a spatial index, the Hierarchical Triangular Mesh;
- the time limit of synchronous queries, and batch jobs that write into a database of your own.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import constants
from matplotlib.patches import Circle, Patch
from SciServer import Authentication, CasJobs, SkyServer

import settings

# The queries before the last section need no SciServer login. Silence
# the warning SciScript gives about the missing login token.
warnings.filterwarnings("ignore", message="In Authentication.getToken")
# Descriptions from the schema are long; show them whole.
pd.set_option("display.max_colwidth", None)

## 1. Sending a query

### What the SDSS measured

The SDSS observes with a $2.5$-meter telescope at Apache Point Observatory in New Mexico. Its database holds two kinds of measurements:

- **Imaging.** A camera scanned the sky in five filters, $u$, $g$, $r$, $i$ and $z$, from the ultraviolet to the near infrared. Software found the objects in the images and measured each one: its position, its brightness in every filter, its size and its shape. These measurements fill the table `PhotoObjAll`, one row per detection.
- **Spectroscopy.** Optical fibers carried the light of selected objects to spectrographs. A spectrum tells what the object is (`class`: star, galaxy or quasar) and its **redshift** $z$, the shift of its spectral lines toward longer wavelengths. For a galaxy the redshift measures its distance, through the expansion of the Universe. The spectra fill `SpecObjAll`, one row per spectrum.

Positions on the sky are given by **right ascension** (`ra`) and **declination** (`dec`), the longitude and latitude of the celestial sphere, both in degrees. One arcminute ($1'$) is $1/60$ of a degree, and one arcsecond ($1''$) is $1/60$ of an arcminute. Brightness is given as a **magnitude**: a logarithmic scale on which larger numbers mean fainter objects, and $5$ magnitudes mean a factor of $100$ in flux.

### SciServer and CasJobs

The SDSS database is served by **SciServer**, a set of web services run by Johns Hopkins University:

- **CasJobs** runs SQL queries on the SDSS databases and keeps a private database, **MyDB**, for each user (section 9);
- **SkyServer** is the web portal of the SDSS, with tools to browse the sky, look up objects, cut out images and read the documentation of the schema;
- **Compute** runs Jupyter notebooks like this one on servers next to the data.

The `SciServer` package imported above is SciScript-Python, the client library of these services. `CasJobs.executeQuery` sends the text of a query to CasJobs over HTTP. CasJobs runs it on the database server, Microsoft SQL Server, and sends the result back as JSON, which SciScript turns into a pandas DataFrame. The call waits for the result, so such a query is called **synchronous**. The SQL dialect is therefore that of SQL Server: `TOP 10` instead of `LIMIT 10`, for example.

The **context** names the database a query runs in. Every SDSS data release is a separate database. These notebooks query DR17 (`settings.CASJOBS_CONTEXT`), the final release of the SDSS-I to SDSS-IV galaxy and quasar surveys. Image cutouts come from DR20 (`settings.SKYSERVER_RELEASE`), the newest SkyServer release; the imaging has not changed since DR17, so both describe the same sky.

### A first query

The first query asks the server about a single position: the center of the Coma cluster, stored in `settings.py`. Three of the database's own functions (section 6) write the position in hours and degrees and turn it into an SDSS object name. The query reads no table, so it returns at once.

In [ ]:
coma_ra, coma_dec = settings.COMA_RA, settings.COMA_DEC
first_query = f"""
SELECT
    dbo.fHMS({coma_ra}) AS ra_hms,
    dbo.fDMS({coma_dec}) AS dec_dms,
    dbo.fIAUFromEq({coma_ra}, {coma_dec}) AS sdss_name
"""
CasJobs.executeQuery(
    sql=first_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

In [ ]:
CasJobs.executeQuery(
    sql=first_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

With `format="json"`, `executeQuery` returns the response as it arrives over the network: the query, and for each result table its column names and rows.

In [ ]:
response = CasJobs.executeQuery(
    sql=first_query, context=settings.CASJOBS_CONTEXT, format="json"
)
print(response)

## 2. How large it is

The table `Diagnostics` stores the number of rows of every table and view (`type` `U` and `V`). Reading it costs nothing, whereas `COUNT(*)` on the largest tables would read every row.

In [ ]:
size_query = """
SELECT name, RTRIM(type) AS type, count AS n_rows
FROM Diagnostics
WHERE type IN ('U', 'V')
"""
sizes = CasJobs.executeQuery(
    sql=size_query, context=settings.CASJOBS_CONTEXT, format="pandas"
).set_index("name")
sizes.sort_values("n_rows", ascending=False).head(10)

The chart shows the three largest tables and the tables and views that these notebooks use. Views (section 4) are drawn in a lighter color.

In [ ]:
shown = [
    "PhotoProfile",
    "Neighbors",
    "PhotoObjAll",
    "PhotoObj",
    "PhotoPrimary",
    "Galaxy",
    "Star",
    "Photoz",
    "SpecObjAll",
    "SpecObj",
    "apogeeStar",
    "aspcapStar",
]
type_colors = {"U": settings.OKABE_ITO[5], "V": settings.OKABE_ITO[2]}
shown_sizes = sizes.loc[shown[::-1]]

with plt.style.context(settings.LIGHT_STYLE):
    fig, ax = plt.subplots(
        figsize=(0.7 * settings.FIGURE_WIDTH, 0.45 * settings.FIGURE_WIDTH)
    )
    ax.barh(
        shown_sizes.index,
        shown_sizes["n_rows"],
        color=shown_sizes["type"].map(type_colors),
    )
    ax.set_xscale("log")
    ax.set_xlabel("rows")
    ax.tick_params(axis="y", which="both", left=False, right=False)
    legend_handles = [
        Patch(color=type_colors["U"], label="table"),
        Patch(color=type_colors["V"], label="view"),
    ]
    ax.legend(handles=legend_handles, loc="lower right")
    plt.show()

What to look for:

- **Billions of rows.** `PhotoObjAll` holds about $1.2$ billion detections. The two largest tables are derived from it: `PhotoProfile` stores the light profile of every object, its average brightness in rings around its center, and `Neighbors` every pair of objects less than $0.5'$ apart.
- **Spectra are rare.** `SpecObjAll` has about $5.8$ million rows, one for about every $200$ detections in the imaging. A spectrum takes far longer to collect than an image.
- **Small tables.** The APOGEE tables of demo 01, `apogeeStar` and `aspcapStar`, hold about $0.7$ million stars, a small table by these standards.

How much data is one table? The function `fDocColumns` (section 3) lists the columns of a table with their size in bytes. Their sum is the width of a row, and the width times the number of rows is the size of the table before compression.

In [ ]:
width_query = """
SELECT COUNT(*) AS n_columns, SUM(length) AS row_bytes
FROM dbo.fDocColumns('PhotoObjAll')
"""
width = CasJobs.executeQuery(
    sql=width_query, context=settings.CASJOBS_CONTEXT, format="pandas"
).iloc[0]
table_bytes = width["row_bytes"] * sizes.loc["PhotoObjAll", "n_rows"]
# A fast network connection, in bytes per second.
download_speed = 100e6
download_hours = table_bytes / download_speed / 3600
print(f"{width['n_columns']} columns, {width['row_bytes']:,} bytes per row")
print(f"PhotoObjAll: {table_bytes / 1e12:.1f} TB")
print(
    f"download at {download_speed / 1e6:.0f} MB/s: {download_hours:.0f} hours"
)

A copy of `PhotoObjAll` alone would take hours to download and terabytes to store, and it is only one of $146$ tables. So the database works the other way around: **the query travels to the data**. The server reads as many rows as the query needs, and only the result crosses the network, usually thousands of rows rather than billions. Every query in these notebooks follows this rule: it selects, filters, joins and computes on the server, and downloads only what a figure needs.

## 3. The database describes itself

A schema of this size needs a map. The SDSS database documents itself in its own tables, so the documentation can be queried like the data:

- `DBObjects` lists every table, view, function and procedure, with a description;
- `DBColumns` lists every column of every table, with its unit and a description;
- `DataConstants` lists the meaning of every flag bit and coded value (section 5).

SkyServer's [Schema Browser](https://skyserver.sdss.org/dr17/MoreTools/browser) shows the same text as web pages.

In [ ]:
object_query = """
SELECT RTRIM(type) AS type, COUNT(*) AS n_objects
FROM DBObjects
GROUP BY type
"""
object_types = {"U": "table", "V": "view", "F": "function", "P": "procedure"}
objects = CasJobs.executeQuery(
    sql=object_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)
objects["type"] = objects["type"].map(object_types)
objects

Most procedures build and check the database when a release is loaded; queries use the tables, views and functions.

`fDocColumns` returns the documentation of one table. Here are the columns of `SpecObjAll` that the demos use:

In [ ]:
column_query = """
SELECT name, type, unit, description
FROM dbo.fDocColumns('SpecObjAll')
WHERE name IN (
    'specObjID', 'bestObjID', 'ra', 'dec', 'z', 'zWarning', 'class',
    'velDisp', 'legacy_target1', 'survey', 'sourceType', 'sciencePrimary'
)
"""
CasJobs.executeQuery(
    sql=column_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

`DBColumns` can be searched like any table. Which tables hold a velocity dispersion, the quantity that demo 01 uses to select standard candles?

In [ ]:
search_query = """
SELECT tablename, COUNT(*) AS n_columns, MIN(name) AS example
FROM DBColumns
WHERE description LIKE '%velocity dispersion%'
GROUP BY tablename
"""
CasJobs.executeQuery(
    sql=search_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

The same quantity appears in eight tables, measured by different pipelines: of stars from the spectra (`SpecObjAll`, used in demo 01), of gas from the emission lines, or within the MaNGA survey's maps of nearby galaxies. The descriptions tell them apart.

Functions are documented too. `DBObjects` holds a description of each, and `fDocFunctionParams` lists its inputs and outputs. Here is `fGetNearbyObjEq`, the cone search of sections 6 and 7:

In [ ]:
function_query = """
SELECT name, description
FROM DBObjects
WHERE name = 'fGetNearbyObjEq'
"""
CasJobs.executeQuery(
    sql=function_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

In [ ]:
parameter_query = """
SELECT name, type, inout
FROM dbo.fDocFunctionParams('fGetNearbyObjEq')
ORDER BY inout, pnum
"""
CasJobs.executeQuery(
    sql=parameter_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

## 4. Tables, views and keys

### One detection or one object

The imaging scans overlap, so much of the sky was imaged more than once, and many objects have several detections in `PhotoObjAll`. The pipeline picks one detection of each object as **primary**; the others are **secondary**. The column `mode` records the choice: $1$ for primary, $2$ for secondary, $3$ for other detections, such as the combined light of objects that overlap. Spectra are repeated too, and `sciencePrimary` in `SpecObjAll` marks the best spectrum at each position.

Most queries should count each object once, so the database offers **views**: stored queries that look like tables. A view holds no rows of its own; the server merges its definition into every query that uses it.

In [ ]:
view_names = [
    "PhotoObj",
    "PhotoPrimary",
    "PhotoSecondary",
    "Galaxy",
    "Star",
    "SpecObj",
]
quoted_names = ", ".join(f"'{name}'" for name in view_names)
view_query = f"""
SELECT name, description, text
FROM DBObjects
WHERE name IN ({quoted_names})
"""
views = CasJobs.executeQuery(
    sql=view_query, context=settings.CASJOBS_CONTEXT, format="pandas"
).set_index("name")
views.insert(0, "n_rows", sizes.loc[views.index, "n_rows"])
views.loc[view_names]

The counts add up: every detection in `PhotoObj` is either primary or secondary.

In [ ]:
primary_and_secondary = sizes.loc[["PhotoPrimary", "PhotoSecondary"], "n_rows"]
print(f"PhotoPrimary + PhotoSecondary: {primary_and_secondary.sum():,}")
print(f"PhotoObj:                      {sizes.loc['PhotoObj', 'n_rows']:,}")

`Galaxy` and `Star` are the rows of `PhotoPrimary` that the imaging classifies as galaxies or stars, and `SpecObj` holds the rows of `SpecObjAll` with `sciencePrimary = 1`. To count objects, use `PhotoPrimary`, `Galaxy` or `Star`, and `SpecObj` for spectra; `PhotoObj` when every detection matters. The demos join `SpecObj` to `PhotoObj`.

### IDs that carry meaning

The primary key of `PhotoObjAll` is `objID`, a $64$-bit integer, and that of `SpecObjAll` is `specObjID`. Both are assembled from the numbers that tell where a measurement comes from:

- An imaging **run** is one continuous scan of the camera. The camera has six columns of detectors (**camcol**), the scan of each column is cut into **fields**, and **obj** numbers the objects in a field.
- A spectrum comes from one **fiber**, plugged into an aluminum **plate** that was observed on a given night, the **MJD** (Modified Julian Date).

Functions unpack the IDs. Below they are applied to the spectrum nearest to the center of Coma and to its object in the imaging; `TOP 1` with `ORDER BY` keeps the nearest of the spectra that the cone search of section 6 returns. The same numbers are also stored as columns, such as `plate`, `mjd` and `fiberID` in `SpecObj`.

In [ ]:
cone = settings.parameter_set("cone_search")
spectra_in_cone = (
    f"dbo.fGetNearbySpecObjEq({coma_ra}, {coma_dec}, {cone['radius']})"
)
id_query = f"""
SELECT TOP 1
    s.specObjID,
    dbo.fPlate(s.specObjID) AS plate,
    dbo.fMJD(s.specObjID) AS mjd,
    dbo.fFiber(s.specObjID) AS fiber,
    s.bestObjID,
    dbo.fRun(s.bestObjID) AS run,
    dbo.fCamcol(s.bestObjID) AS camcol,
    dbo.fField(s.bestObjID) AS field,
    dbo.fObj(s.bestObjID) AS obj
FROM {spectra_in_cone} AS nearby
    JOIN SpecObj AS s ON s.specObjID = nearby.specObjID
ORDER BY nearby.distance
"""
CasJobs.executeQuery(
    sql=id_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

### Keys and indexes

`IndexMap` lists the keys and indexes of every table: primary keys (`K`), foreign keys (`F`) and ordinary indexes (`I`). An index is a sorted copy of some columns that lets the server find rows without reading the whole table. The SDSS indexes carry the most-used columns along (`fieldList` names them all), so many queries are answered from an index alone, reading a few dozen bytes per row instead of $2$ kilobytes.

In [ ]:
index_query = """
SELECT tableName, code, type, fieldList, foreignKey
FROM IndexMap
WHERE tableName IN ('PhotoObjAll', 'SpecObjAll')
ORDER BY tableName, code
"""
CasJobs.executeQuery(
    sql=index_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

Several indexes start with `htmID`, a number that encodes the position on the sky. Section 7 shows how the cone searches use it.

### Spectra and images

Every demo joins spectra to imaging: the redshift comes from `SpecObj`, the brightness and size from `PhotoObj`. The link is the column `bestObjID` of `SpecObj`, the `objID` of the object imaged at the position of the fiber. Demos 01 to 03 also join `Photoz`, the table of photometric redshift estimates, which holds the k-corrections as well. `Photoz` should have a row for every galaxy, but a few are missing. A plain `JOIN` drops a spectrum without a match; a `LEFT JOIN` keeps it and fills the columns from `Photoz` with `NULL`. `COUNT(column)` counts only the values that are not `NULL`, so one query counts both cases. `spectra_in_cone`, defined above, is the cone search of section 6: it limits the query to the spectra around Coma.

In [ ]:
photoz_query = f"""
SELECT
    COUNT(*) AS galaxies,
    COUNT(pz.objID) AS with_photoz
FROM {spectra_in_cone} AS nearby
    JOIN SpecObj AS s ON s.specObjID = nearby.specObjID
    JOIN PhotoObj AS p ON p.objID = s.bestObjID
    LEFT JOIN Photoz AS pz ON pz.objID = s.bestObjID
WHERE s.class = 'GALAXY'
"""
print(photoz_query)
CasJobs.executeQuery(
    sql=photoz_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

Whether to keep the galaxies without `Photoz` depends on the question. Demo 03 keeps them, because dropping them would punch holes in its map; demos 01 and 02 need the k-corrections, so they use a plain `JOIN`.

## 5. Flags, codes and missing values

### Bit flags

Many columns pack yes-or-no answers into the bits of one integer. The `flags` column of `PhotoObj` has $64$ bits, one for each problem or property the imaging pipeline can report: saturated pixels, an object at the edge of an image, a blend of overlapping objects, and so on. Bit $18$, `SATURATED`, has the value $2^{18} = 262144$. A bit is tested with the bitwise AND operator `&`: `(flags & 262144) > 0` holds where it is set.

A bare number like this is easy to get wrong, so every family of flags comes with a pair of functions: `fPhotoFlags('SATURATED')` returns the value of a named bit, and `fPhotoFlagsN(flags)` lists the names of all bits that are set. The view `PhotoFlags` lists all $64$ bits with their descriptions:

In [ ]:
photo_flags = CasJobs.executeQuery(
    sql="SELECT name, value, description FROM PhotoFlags",
    context=settings.CASJOBS_CONTEXT,
    format="pandas",
).set_index("name")
print(f"{len(photo_flags)} bits")
photo_flags.loc[["EDGE", "BLENDED", "SATURATED"]]

The brightest objects near the center of Coma show many of these bits. The column `clean` condenses them into one value, $1$ for photometry that the pipeline judges clean. The procedure that sets it, `spSetPhotoClean`, documents conditions chosen for point sources and warns that "the same logic is not applicable to extended objects". Among galaxies, demo 01 finds that the unclean ones mostly touch the edge of an image, contain saturated pixels or overlap a neighbor.

In [ ]:
flag_limits = settings.parameter_set("flag_examples")
objects_near_center = (
    f"dbo.fGetNearbyObjEq({coma_ra}, {coma_dec}, {flag_limits['radius']})"
)
bright_query = f"""
SELECT TOP {flag_limits["size"]}
    p.r,
    dbo.fPhotoTypeN(p.type) AS type,
    p.clean,
    CASE
        WHEN (p.flags & dbo.fPhotoFlags('SATURATED')) > 0 THEN 1 ELSE 0
    END AS saturated,
    dbo.fPhotoFlagsN(p.flags) AS flag_names
FROM {objects_near_center} AS nearby
    JOIN PhotoObj AS p ON p.objID = nearby.objID
ORDER BY p.r
"""
print(bright_query)
CasJobs.executeQuery(
    sql=bright_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

In a field as crowded as the center of a cluster, many galaxies are not clean: bright neighbors, blends and saturated stars spoil their measurements.

In [ ]:
clean_query = f"""
SELECT g.clean, COUNT(*) AS galaxies
FROM {objects_near_center} AS nearby
    JOIN Galaxy AS g ON g.objID = nearby.objID
GROUP BY g.clean
"""
CasJobs.executeQuery(
    sql=clean_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

Demo 02 requires `p.clean = 1`; demo 01 explains why it does not.

### Target bits

Why was a spectrum taken? The SDSS chose its spectroscopic targets from the imaging by fixed rules, and the **target bits** record which rules an object passed. For SDSS-I/II they are stored in `legacy_target1`. Its bit $6$, with value $64$, is `GALAXY`: the Main Galaxy Sample of all three demos. `fPrimTarget` and `fPrimTargetN` translate between names and values, like `fPhotoFlags` and `fPhotoFlagsN`. Here are the spectra around Coma, grouped by their target bits:

In [ ]:
target_query = f"""
SELECT
    s.legacy_target1,
    dbo.fPrimTargetN(s.legacy_target1) AS target_names,
    COUNT(*) AS spectra
FROM {spectra_in_cone} AS nearby
    JOIN SpecObj AS s ON s.specObjID = nearby.specObjID
GROUP BY s.legacy_target1
ORDER BY spectra DESC
"""
CasJobs.executeQuery(
    sql=target_query, context=settings.CASJOBS_CONTEXT, format="pandas"
).head(8)

An object can pass several rules: `GALAXY GALAXY_RED` ($64 + 32 = 96$) is a galaxy of the Main Galaxy Sample that also passed the selection of luminous red galaxies. Spectra with no bit set are either sky and calibration fibers of SDSS-I/II, whose bits are in `legacy_target2`, or spectra of the later surveys, BOSS, eBOSS and SEGUE, which record their targets in other columns (`boss_target1`, `eboss_target1`, `segue2_target1`, ...), each with its own pair of functions. Demo 01 uses `fApogeeExtraTarg` for the APOGEE survey of stars.

### Coded values

Other columns hold codes. `type` in `PhotoObj` is a number ($3$ for a galaxy, $6$ for a star; `fPhotoTypeN` names it), while `class`, `survey` and `sourceType` in `SpecObj` are short strings: `class` is what the spectrum turned out to be, `sourceType` what the object was targeted as, and `survey` the survey that took the spectrum. Demo 03 selects its samples with these three columns. They do not always agree. Some quasar candidates turned out to be stars. And the fit gives a `class` to every spectrum, even to the fibers placed on empty sky for calibration (`sourceType` `SKY`); their redshifts are fits to noise, which the quality flag below catches.

In [ ]:
class_query = f"""
SELECT s.class, s.survey, s.sourceType, COUNT(*) AS spectra
FROM {spectra_in_cone} AS nearby
    JOIN SpecObj AS s ON s.specObjID = nearby.specObjID
GROUP BY s.class, s.survey, s.sourceType
ORDER BY spectra DESC
"""
CasJobs.executeQuery(
    sql=class_query, context=settings.CASJOBS_CONTEXT, format="pandas"
).head(12)

### Quality of a redshift

`zWarning` is a bitmask as well: $0$ means that the redshift fit found no problem, and every demo requires `zWarning = 0`. `fSpecZWarningN` names the bits.

In [ ]:
warning_query = f"""
SELECT
    s.zWarning,
    dbo.fSpecZWarningN(s.zWarning) AS warning_names,
    COUNT(*) AS spectra
FROM {spectra_in_cone} AS nearby
    JOIN SpecObj AS s ON s.specObjID = nearby.specObjID
GROUP BY s.zWarning
ORDER BY spectra DESC
"""
CasJobs.executeQuery(
    sql=warning_query, context=settings.CASJOBS_CONTEXT, format="pandas"
).head(6)

`SMALL_DELTA_CHI2` means that a second redshift fits the spectrum almost as well as the best one, so the redshift is uncertain; `SKY` marks a fiber that was placed on empty sky.

### Missing values

SQL marks a missing value with `NULL`, but many astronomical catalogs use an impossible number instead, most often $-9999$. The APOGEE tables of demo 01 do this: a star whose temperature could not be measured has `teff = -9999`. To the database this is an ordinary number, and it enters every average. `NULLIF(teff, -9999)` turns it into `NULL`, which aggregates skip and pandas reads as `NaN`:

In [ ]:
sentinel_query = """
SELECT
    COUNT(*) AS stars,
    COUNT(NULLIF(teff, -9999)) AS measured,
    AVG(teff) AS mean_teff,
    AVG(NULLIF(teff, -9999)) AS mean_measured_teff
FROM aspcapStar
"""
CasJobs.executeQuery(
    sql=sentinel_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

The markers make up about $6\,\%$ of the rows, yet they pull the mean temperature down by about $900$ kelvin.

## 6. Functions inside queries

The sections above have already used more than a dozen of the database's functions. They come in the two kinds that SQL Server, like most databases, supports:

- **Scalar functions** return one value. They go wherever a value can go, in the `SELECT` list, in `WHERE` or in `ORDER BY`, and run once for every row: `fPhotoFlagsN(p.flags)`, `fRun(s.bestObjID)`.
- **Table-valued functions** return a table. They go in `FROM` and join like a table: `fDocColumns('SpecObjAll')`, `fGetNearbySpecObjEq(ra, dec, r)`.

SQL Server requires the schema name, `dbo.` (for "database owner"), in front of every scalar function; the notebooks write it for all functions. The families seen so far handle documentation (`fDocColumns`, `fDocFunctionParams`), flags and codes (`fPhotoFlags`, `fPrimTarget`, `fPhotoTypeN`, ...) and IDs (`fRun`, `fPlate`, ...). Two more families do work that would be hard to write in SQL: geometry on the sky and cosmology.

### A cone search around Coma

`fGetNearbySpecObjEq(ra, dec, r)` returns every spectrum within $r$ arcminutes of a position, with its angular distance from the center in the column `distance`. The query below joins `SpecObj` for the redshift and class and `PhotoObj` for the brightness. Two scalar functions compute columns on the server: `fIAUFromEq` the SDSS name of each galaxy, and `fCosmoDistanceModulus` the distance modulus of its redshift. From the distance modulus follows the absolute magnitude, as in demo 02, here without the k-correction, which is small at the redshift of Coma.

In [ ]:
cosmology = settings.COSMOLOGY
radiation_density = cosmology.Ogamma0 + cosmology.Onu0
distance_modulus = (
    f"dbo.fCosmoDistanceModulus(s.z, {cosmology.Om0}, {cosmology.Ode0}, "
    f"{radiation_density}, {cosmology.w(0)}, {cosmology.h})"
)
cone_query = f"""
SELECT
    dbo.fIAUFromEq(s.ra, s.dec) AS sdss_name,
    s.ra, s.dec, s.z, nearby.distance,
    p.dered_r,
    p.dered_r - {distance_modulus} AS abs_r
FROM {spectra_in_cone} AS nearby
    JOIN SpecObj AS s ON s.specObjID = nearby.specObjID
    JOIN PhotoObj AS p ON p.objID = s.bestObjID
WHERE
    s.class = 'GALAXY'
    AND s.zWarning = 0
    AND s.z > 0
"""
print(cone_query)

In [ ]:
%%time
galaxies = CasJobs.executeQuery(
    sql=cone_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)
print(f"{len(galaxies):,} galaxies")

The left panel below is the histogram of their redshifts. The right panel places them on the sky around the center, in arcminutes east and north; east is to the left, as on a map of the sky seen from below. The galaxies in the shaded range of redshift are drawn in color.

In [ ]:
def sky_offsets(ra, dec):
    """
    Offsets east and north of the center of Coma, in arcminutes.

    A flat map of the sky around the center, accurate to about one
    percent within a degree.
    """
    east = (ra - coma_ra) * np.cos(np.deg2rad(coma_dec)) * 60
    north = (dec - coma_dec) * 60
    return east, north


members = galaxies["z"].between(cone["member_z_min"], cone["member_z_max"])
east, north = sky_offsets(galaxies["ra"], galaxies["dec"])
z_bins = np.arange(0, cone["z_max"] + cone["z_step"], cone["z_step"])
member_color = settings.OKABE_ITO[6]

with plt.style.context(settings.LIGHT_STYLE):
    fig, (hist_ax, sky_ax) = plt.subplots(
        1, 2, figsize=(settings.FIGURE_WIDTH, 0.47 * settings.FIGURE_WIDTH)
    )
    hist_ax.axvspan(
        cone["member_z_min"],
        cone["member_z_max"],
        color=member_color,
        alpha=0.2,
        linewidth=0,
    )
    hist_ax.hist(galaxies["z"], bins=z_bins, histtype="step")
    hist_ax.set_xlim(0, cone["z_max"])
    hist_ax.set_xlabel("redshift $z$")
    hist_ax.set_ylabel("galaxies")
    sky_ax.scatter(
        east[~members], north[~members], s=3, color="0.6", label="others"
    )
    sky_ax.scatter(
        east[members],
        north[members],
        s=3,
        color=member_color,
        label="shaded redshifts",
    )
    edge = Circle(
        (0, 0),
        cone["radius"],
        fill=False,
        edgecolor=plt.rcParams["axes.edgecolor"],
    )
    sky_ax.add_patch(edge)
    sky_ax.set_aspect("equal")
    sky_ax.invert_xaxis()
    sky_ax.set_xlabel("east (arcmin)")
    sky_ax.set_ylabel("north (arcmin)")
    sky_ax.legend(
        loc="lower center", bbox_to_anchor=(0.5, 1), ncols=2, markerscale=3
    )
    plt.show()

In [ ]:
member_z = galaxies.loc[members, "z"]
speed_of_light = constants.c.to("km/s").value
# Half the distance between the 16th and 84th percentiles: the standard
# deviation of a normal distribution, insensitive to outliers.
spread = (member_z.quantile(0.84) - member_z.quantile(0.16)) / 2
print(f"{members.sum()} galaxies, median redshift {member_z.median():.4f}")
print(f"recession velocity cz: {speed_of_light * member_z.median():,.0f} km/s")
print(f"spread of cz:          {speed_of_light * spread:,.0f} km/s")

What to look for:

- **The cluster.** Most of the galaxies in the cone fall in a narrow spike of redshift: the Coma cluster. Its median redshift corresponds to a recession velocity $cz$ of about $7000\,\mathrm{km\,s^{-1}}$, and its galaxies spread around it by about $1000\,\mathrm{km\,s^{-1}}$: their orbital speeds inside the cluster.
- **Concentration.** The cluster's galaxies crowd toward the center, while the galaxies in front of and behind the cluster spread over the whole cone.

The most luminous of them are the giants of the cluster:

In [ ]:
galaxies[members].nsmallest(5, "abs_r")[
    ["sdss_name", "z", "dered_r", "abs_r", "distance"]
]

The two brightest, a few arcminutes from the center, are NGC 4889 and NGC 4874, the giant elliptical galaxies at the heart of the cluster.

### Pictures from the same sky

SkyServer draws image cutouts from the imaging that the catalog was measured on; demo 02 builds its galleries this way. `SkyServer.getJpegImgCutout` returns a color JPEG of any patch of sky: `scale` is the size of a pixel in arcseconds, `width` and `height` the size of the image in pixels. North is up and east to the left. The circles mark the galaxies of the shaded redshift range from the table above: the rows of the table are the galaxies in the picture.

In [ ]:
%%time
field_of_view = cone["cutout_field"]
cutout_width = cone["cutout_width"]
coma_image = SkyServer.getJpegImgCutout(
    ra=coma_ra,
    dec=coma_dec,
    scale=field_of_view * 60 / cutout_width,
    width=cutout_width,
    height=cutout_width,
    dataRelease=settings.SKYSERVER_RELEASE,
)

In [ ]:
half_field = field_of_view / 2
in_field = members & (east.abs() < half_field) & (north.abs() < half_field)

with plt.style.context(settings.DARK_STYLE):
    fig, ax = plt.subplots(
        figsize=(0.65 * settings.FIGURE_WIDTH, 0.65 * settings.FIGURE_WIDTH)
    )
    # The image's left edge lies east of the center.
    ax.imshow(
        coma_image, extent=(half_field, -half_field, -half_field, half_field)
    )
    ax.scatter(
        east[in_field],
        north[in_field],
        s=120,
        facecolors="none",
        edgecolors=settings.OKABE_ITO[2],
        linewidths=0.8,
    )
    ax.set_xlabel("east (arcmin)")
    ax.set_ylabel("north (arcmin)")
    plt.show()

## 7. Why the cone search is fast

### A scan against an index

The cone search can also be written with a scalar function alone: compute the distance of every spectrum from the center with `fDistanceArcMinEq` and keep those within the radius. The answer is the same, but the server has to compute the distance for every row of the table, because no index can find the rows near a point from a formula. On `SpecObj`, with about $5$ million rows, both versions finish:

In [ ]:
scan = settings.parameter_set("scan_comparison")
spectra_radius = scan["spectra_radius"]
spectra_scan_query = f"""
SELECT COUNT(*) AS spectra
FROM SpecObj AS s
WHERE
    dbo.fDistanceArcMinEq(s.ra, s.dec, {coma_ra}, {coma_dec})
        < {spectra_radius}
"""
spectra_index_query = f"""
SELECT COUNT(*) AS spectra
FROM dbo.fGetNearbySpecObjEq({coma_ra}, {coma_dec}, {spectra_radius}) AS n
    JOIN SpecObj AS s ON s.specObjID = n.specObjID
"""
print(spectra_scan_query)
print(spectra_index_query)

In [ ]:
%%time
CasJobs.executeQuery(
    sql=spectra_scan_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

In [ ]:
%%time
CasJobs.executeQuery(
    sql=spectra_index_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

`Galaxy` has $40$ times as many rows as `SpecObj`. Through the function, the search takes about as long as before:

In [ ]:
galaxy_radius = scan["galaxy_radius"]
galaxy_index_query = f"""
SELECT COUNT(*) AS galaxies
FROM dbo.fGetNearbyObjEq({coma_ra}, {coma_dec}, {galaxy_radius}) AS n
    JOIN Galaxy AS g ON g.objID = n.objID
"""
galaxy_scan_query = f"""
SELECT COUNT(*) AS galaxies
FROM Galaxy AS g
WHERE
    dbo.fDistanceArcMinEq(g.ra, g.dec, {coma_ra}, {coma_dec})
        < {galaxy_radius}
"""

In [ ]:
%%time
CasJobs.executeQuery(
    sql=galaxy_index_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

The scan, however, runs into the time limit of synchronous queries, about a minute, and CasJobs stops it. The cell below catches the error and prints its message; it takes a minute to run.

In [ ]:
%%time
try:
    CasJobs.executeQuery(
        sql=galaxy_scan_query,
        context=settings.CASJOBS_CONTEXT,
        format="pandas",
    )
# SciScript reports every failed request as a plain Exception.
except Exception as error:  # noqa: BLE001
    print(error)

### The Hierarchical Triangular Mesh

How does the function find the rows near a point without reading the table? Through the column `htmID`, which `IndexMap` showed at the front of several indexes. The **Hierarchical Triangular Mesh** (HTM) divides the sky into $8$ spherical triangles, divides each of them into $4$ smaller ones, and repeats. The SDSS goes $20$ levels deep: $8 \times 4^{20} \approx 9 \times 10^{12}$ triangles, each about $0.4''$ across, the size of a pixel of the camera. The `htmID` of an object encodes the path of triangles down to the smallest one that contains it. `fHtmEq` computes it for a position, and `fHtmGetString` writes it as a path: the hemisphere (`N` or `S`), the starting triangle, and one digit from $0$ to $3$ for each of the $20$ levels.

In [ ]:
htm_query = f"""
SELECT
    dbo.fHtmEq({coma_ra}, {coma_dec}) AS htm_id,
    dbo.fHtmGetString(dbo.fHtmEq({coma_ra}, {coma_dec})) AS path
"""
CasJobs.executeQuery(
    sql=htm_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

Triangles inside the same parent share the beginning of their path, so their IDs form one interval. The objects of any patch of sky therefore fill a few intervals of `htmID`, and because the table is indexed by `htmID`, each interval is a contiguous slice of the index that the server can read directly. A search in two dimensions becomes a few range lookups in one. `fHtmCoverCircleEq(ra, dec, r)` returns intervals whose triangles together cover a circle:

In [ ]:
htm = settings.parameter_set("htm_cover")
cover_query = f"""
SELECT HtmIDStart, HtmIDEnd
FROM dbo.fHtmCoverCircleEq({coma_ra}, {coma_dec}, {htm["radius"]})
"""
cover = CasJobs.executeQuery(
    sql=cover_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)
cover

Joining the intervals to `Galaxy` with `BETWEEN` returns every galaxy in the covering triangles. How to run the join is up to the server's query planner, and for the query below it finds a fast plan by itself. Small changes can mislead it: add `TOP 100000` and the condition `g.clean = 1`, and the planner picks a plan that runs into the time limit. `INNER LOOP JOIN` is a join hint of SQL Server that prevents this. It forces the server to loop over the intervals and look up each one in the `htmID` index, and with it the same query takes seconds. The query below carries the hint as well. The triangles cover more than the circle, so the exact distance from `fDistanceArcMinEq` decides which candidates lie inside.

In [ ]:
candidate_query = f"""
SELECT
    htm.HtmIDStart AS interval_start,
    g.ra,
    g.dec,
    dbo.fDistanceArcMinEq(g.ra, g.dec, {coma_ra}, {coma_dec}) AS distance
FROM dbo.fHtmCoverCircleEq({coma_ra}, {coma_dec}, {htm["radius"]}) AS htm
    INNER LOOP JOIN Galaxy AS g
        ON g.htmID BETWEEN htm.HtmIDStart AND htm.HtmIDEnd
"""
print(candidate_query)

In [ ]:
%%time
candidates = CasJobs.executeQuery(
    sql=candidate_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)
inside = (candidates["distance"] < htm["radius"]).to_numpy()
print(f"{len(candidates):,} candidates, {inside.sum():,} inside the circle")

`fGetNearbyObjEq` returns the same galaxies:

In [ ]:
nearby_count_query = f"""
SELECT COUNT(*) AS galaxies
FROM dbo.fGetNearbyObjEq({coma_ra}, {coma_dec}, {htm["radius"]}) AS n
    JOIN Galaxy AS g ON g.objID = n.objID
"""
CasJobs.executeQuery(
    sql=nearby_count_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)

The map shows the candidates, colored by the interval of `htmID` they were found in. The candidates outside the circle are faded.

In [ ]:
interval_number = candidates["interval_start"].rank(method="dense") - 1
interval_colors = plt.get_cmap("tab20")(interval_number.astype(int) % 20)
candidate_east, candidate_north = sky_offsets(
    candidates["ra"].to_numpy(), candidates["dec"].to_numpy()
)

with plt.style.context(settings.LIGHT_STYLE):
    fig, ax = plt.subplots(
        figsize=(0.6 * settings.FIGURE_WIDTH, 0.6 * settings.FIGURE_WIDTH)
    )
    for selected, alpha in [(~inside, 0.25), (inside, 1)]:
        ax.scatter(
            candidate_east[selected],
            candidate_north[selected],
            c=interval_colors[selected],
            s=1,
            alpha=alpha,
            linewidths=0,
            rasterized=True,
        )
    edge = Circle(
        (0, 0),
        htm["radius"],
        fill=False,
        edgecolor=plt.rcParams["axes.edgecolor"],
    )
    ax.add_patch(edge)
    ax.set_aspect("equal")
    ax.invert_xaxis()
    ax.set_xlabel("east (arcmin)")
    ax.set_ylabel("north (arcmin)")
    plt.show()

What to look for:

- **Triangles.** Each color is one interval of `htmID`: one triangle of the mesh, or a few neighboring triangles whose IDs follow each other. Straight edges separate the colors, and the faded galaxies end along straight lines.
- **Overshoot.** The triangles reach well beyond the circle, and the cover returns nearly twice as many galaxies as the circle holds. The faded ones are read from the index and then dropped by the distance cut. That is the price of indexing a two-dimensional sky with one number, and it is small next to reading the whole table.
- **What the function hides.** `fGetNearbyObjEq` does the same work, the cover, the lookups in the index and the distance cut, and returns the same galaxies.

## 8. Habits on a shared server

CasJobs serves many users from the same servers. A few rules and habits keep queries fast:

- **The time limit.** A synchronous query (`executeQuery`) that runs longer than about a minute is stopped, as in section 7. Longer queries go to the batch queue (section 9).
- **Count first.** `SELECT COUNT(*)` with the same `FROM` and `WHERE` reads the same rows but returns a single number, which tells how large the download will be. Demo 03 counts each of its samples before downloading it.
- **Peek with `TOP`.** `SELECT TOP 10 ...` returns ten rows, but not ten random rows: without `ORDER BY`, the server returns whichever rows its plan reaches first. Demo 02 draws a random sample with `ORDER BY NEWID()`.
- **Ask for the columns you need.** `SELECT *` on `PhotoObj` returns $509$ columns, about $2$ kilobytes per row.
- **Build queries from parameters.** Every query here is a Python f-string, filled with values from `settings.py` and `parameter_sets.json` and printed before it runs. CasJobs takes nothing but the text of a query, so this is how a query gets its parameters. It is safe because the values come from our own files; pasting text typed by other people into SQL would open the door to SQL injection.
- **Compute on the server.** Arithmetic, functions, joins and `GROUP BY` run next to the data, and only the result travels. The redshift histogram of section 6, for example, can be counted by the server, which then sends one row per bin instead of one row per galaxy:

In [ ]:
z_step = cone["z_step"]
histogram_query = f"""
SELECT
    FLOOR(s.z / {z_step}) * {z_step} AS z_low,
    COUNT(*) AS galaxies
FROM {spectra_in_cone} AS nearby
    JOIN SpecObj AS s ON s.specObjID = nearby.specObjID
WHERE
    s.class = 'GALAXY'
    AND s.zWarning = 0
    AND s.z > 0
    AND s.z < {cone["z_max"]}
GROUP BY FLOOR(s.z / {z_step})
ORDER BY z_low
"""
histogram = CasJobs.executeQuery(
    sql=histogram_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)
print(f"{len(histogram)} rows instead of {len(galaxies):,}")
histogram.head(15)

## 9. Batch jobs and MyDB

Synchronous queries stop after a minute and send their result straight back. **Batch jobs** are the other way to run a query, and the one CasJobs is named after (Catalog Archive Server Jobs). A job waits in a queue, may run much longer, and writes its result into a table in **MyDB**, a database of your own on the CasJobs servers. The table stays there: later queries read it at once, join it with the SDSS tables, or download it.

MyDB belongs to a SciServer account, so this section needs a login. On SciServer Compute the notebook is logged in already. On your own computer, the cell below asks for your SciServer user name and password.

In [ ]:
if Authentication.getToken() is None:
    Authentication.login()
user = Authentication.getKeystoneUserWithToken(Authentication.getToken())
print(f"logged in as {user.userName}")

The job below stores the sample that all three demos start from: the spectra of the Main Galaxy Sample, with the photometry and the k-corrections that the demos join to them. `INTO mydb.main_galaxies` names the table to create. The job runs in the DR17 context, where the SDSS tables are, and the prefix `mydb.` sends its result to MyDB.

In [ ]:
job = settings.parameter_set("batch_job")
job_query = f"""
SELECT
    s.specObjID, s.ra, s.dec, s.z,
    p.dered_g, p.dered_r, p.petroR90_r, p.clean,
    pz.kcorrG, pz.kcorrR
INTO mydb.{job["table"]}
FROM SpecObj AS s
    JOIN PhotoObj AS p ON p.objID = s.bestObjID
    LEFT JOIN Photoz AS pz ON pz.objID = s.bestObjID
WHERE
    (s.legacy_target1 & dbo.fPrimTarget('GALAXY')) > 0
    AND s.class = 'GALAXY'
    AND s.zWarning = 0
    AND s.z < {job["z_max"]}
"""
print(job_query)

A job cannot create a table that already exists, so the cell first drops the table that an earlier run of this notebook left behind. `submitJob` puts the query in the queue and returns the number of the job at once.

In [ ]:
my_tables = [table["Name"] for table in CasJobs.getTables("MyDB")]
if job["table"] in my_tables:
    drop_job = CasJobs.submitJob(f"DROP TABLE {job['table']}", context="MyDB")
    CasJobs.waitForJob(drop_job)
job_id = CasJobs.submitJob(job_query, context=settings.CASJOBS_CONTEXT)
print(f"job {job_id} submitted")

`waitForJob` asks for the status of the job every few seconds until it ends. `Status` $5$ means finished, $4$ failed (`Message` says why) and $3$ canceled. The CasJobs web site lists the job as well.

In [ ]:
%%time
status = CasJobs.waitForJob(job_id)
{key: status[key] for key in ["Status", "Message", "TimeStart", "TimeEnd"]}

The new table is listed among the tables of MyDB:

In [ ]:
pd.DataFrame(CasJobs.getTables("MyDB"))

Queries on the table run in the MyDB context. The table holds only the Main Galaxy Sample, so the query below reads a few hundred thousand rows instead of joining three tables of hundreds of millions, and it returns in a moment:

In [ ]:
%%time
job_z_step = job["z_step"]
mydb_query = f"""
SELECT
    FLOOR(z / {job_z_step}) * {job_z_step} AS z_low,
    COUNT(*) AS galaxies,
    AVG(dered_g - dered_r) AS mean_observed_gr
FROM {job["table"]}
GROUP BY FLOOR(z / {job_z_step})
ORDER BY z_low
"""
by_redshift = CasJobs.executeQuery(
    sql=mydb_query, context="MyDB", format="pandas"
)
by_redshift

The table stays in MyDB until you drop it, and later queries can start from it instead of repeating the join. CasJobs can also upload a table of your own into MyDB (`CasJobs.uploadPandasDataFrameToTable`), such as a list of positions from another catalog, to join with the SDSS tables in a job.

## 10. Where the demos use this

The demo notebooks are:

- 01, *Hubble–Lemaître and HR diagrams*;
- 02, *Galaxy gallery*;
- 03, *Map of the Universe*.

| Idea | Section here | Demos |
| --- | --- | --- |
| `executeQuery`, the DR17 context | 1 | all |
| `SpecObj` joined to `PhotoObj` through `bestObjID` | 4 | all |
| `JOIN` and `LEFT JOIN` to `Photoz` | 4 | 01 and 02 (`JOIN`), 03 (`LEFT JOIN`) |
| target bits, `fPrimTarget('GALAXY')` and `fApogeeExtraTarg('DUPLICATE')` | 5 | all; 01 |
| `zWarning` and `class` | 5 | all |
| `clean` | 5 | 02 (01 explains why not) |
| `survey` and `sourceType` | 5 | 03 |
| the APOGEE tables `apogeeStar` and `aspcapStar` | 2, 5 | 01 |
| $-9999$ and `NULLIF` | 5 | 01 |
| columns computed in the query | 6 | 01, 02 |
| `fCosmoDistanceModulus` | 6 | 02 |
| `SkyServer.getJpegImgCutout` and the DR20 images | 1, 6 | 02 |
| `TOP` and random samples | 8 | 02 |
| `COUNT(*)` before downloading | 8 | 03 |
| queries built from parameters | 8 | all |
